# Rossmann Controlling — Plan/Actual, Variance & Profitability

**Portfolio project 09 · a controlling-oriented companion to
[project 08](../08_rossmann_sales_analysis/).**

This notebook takes the real Kaggle *Rossmann Store Sales* data (1,017,209 rows,
1,115 stores, 2013-01-01 → 2015-07-31) and builds a small **controlling
cockpit**: a plan/actual comparison, a variance analysis (absolute and %), and a
cost/profit model based on **contribution-margin accounting**
(*Deckungsbeitragsrechnung*).

> ### ⚠️ Honesty note — please read first
> The Kaggle dataset contains **only** `Sales`, `Customers`, `Promo`,
> `StoreType` etc. It has **no budget, cost or margin data whatsoever.**
> Therefore:
> * The **plan** is *derived* from the data (prior-year baseline + an assumed
>   growth target) — it is a transparent planning rule, not an official budget.
> * All **cost and margin** figures are **explicit modelling assumptions**
>   (clearly flagged as `MODEL ASSUMPTIONS`), used to demonstrate the *method*.
> * **None** of the euro cost/profit values are real Rossmann company figures.
>
> The goal is to show *controlling method and reasoning* on real sales data —
> honestly, without inventing facts. See §7 for the full limitations list.

## 0 · Setup, Configuration & Model Assumptions

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (12, 5)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# --- Paths -------------------------------------------------------------------
DATA_DIR   = "./data"
OUTPUT_DIR = "./output"
FIG_DIR    = "./figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# --- Analysis window ---------------------------------------------------------
# 2013 is used ONLY as the planning baseline for 2014, and 2014 as the baseline
# for 2015. The plan/actual and P&L analyses therefore run on FY2014 and
# FY2015-YTD (the raw data ends 2015-07-31).
PLAN_YEARS = [2014, 2015]

MAP_ASSORT = {"a": "Basic", "b": "Extra", "c": "Extended"}

# =============================================================================
#  MODEL ASSUMPTIONS  --  NOT real Rossmann figures.
#  The dataset has no budget/cost data; every value below is an explicit,
#  transparent assumption for demonstration only. Change them in one place here
#  and re-run to see the whole cockpit update.
# =============================================================================
GROWTH_TARGET = 0.03            # planned YoY sales growth vs. prior year (+3 %)

VAR_COST_RATIO = {             # variable cost (COGS + variable opex) as a share
    "a": 0.80,                 # of net sales, differentiated by store type to
    "b": 0.83,                 # reflect assortment mix (broader assortment ->
    "c": 0.79,                 # more low-margin volume). Base 80 %, +/-1-3 pp.
    "d": 0.81,                 # Illustrative spreads only.
}
FIXED_COST_PER_OPEN_DAY = 900  # EUR per store per open trading day
                               # (rent, base staffing, utilities)

ASSUMPTIONS = [
    ("Planned YoY growth target", f"+{GROWTH_TARGET:.0%}",
     "Applied to each store's prior-year avg. daily sales to set its plan."),
    ("Variable cost ratio - type a (Basic)", f"{VAR_COST_RATIO['a']:.0%} of sales",
     "COGS + variable operating cost."),
    ("Variable cost ratio - type b (Extra)", f"{VAR_COST_RATIO['b']:.0%} of sales",
     "Broader assortment -> more low-margin volume (illustrative)."),
    ("Variable cost ratio - type c (Extended)", f"{VAR_COST_RATIO['c']:.0%} of sales",
     "Leaner assortment (illustrative)."),
    ("Variable cost ratio - type d", f"{VAR_COST_RATIO['d']:.0%} of sales",
     "COGS + variable operating cost."),
    ("Fixed cost per open trading day", f"EUR {FIXED_COST_PER_OPEN_DAY:,}",
     "Per store per open day: rent, base staffing, utilities."),
]
print("Setup complete. Analysis window (plan years):", PLAN_YEARS)
print("Model assumptions loaded — see the ASSUMPTIONS table (exported in §6).")

## 1 · Load Data

In [ ]:
def find_file(fname, base):
    target = fname.lower()
    for root, _, files in os.walk(base):
        for f in files:
            if f.lower() == target:
                return os.path.join(root, f)
    return None

FILES = {"train": "train.csv", "store": "store.csv"}
raw = {}
for key, fname in FILES.items():
    path = find_file(fname, DATA_DIR)
    if path is None:
        raise FileNotFoundError(
            f"Required file missing: {fname} in {DATA_DIR}. "
            "Download train.csv & store.csv from the Kaggle 'Rossmann Store "
            "Sales' competition and place them in ./data (see README)."
        )
    kw = {"dtype": {"StateHoliday": "str"}} if key == "train" else {}
    raw[key] = pd.read_csv(path, low_memory=False, **kw)
    print(f"[loaded]  {fname:12s} {raw[key].shape[0]:>9,} rows x {raw[key].shape[1]} cols")

## 2 · Prepare, Merge & Scope

We add calendar fields, join the store master (for `StoreType`) and restrict to
**actual trading days** — rows where the store was open with non-zero sales —
exactly the scope used in project 08.

In [ ]:
train = raw["train"].copy()
train["Date"] = pd.to_datetime(train["Date"], errors="coerce")
train["Year"] = train["Date"].dt.year
train["Month"] = train["Date"].dt.month
train["YearMonth"] = train["Date"].dt.to_period("M").astype(str)

store = raw["store"][["Store", "StoreType", "Assortment"]].copy()
store["Assortment_lbl"] = store["Assortment"].map(MAP_ASSORT).fillna(store["Assortment"])

master = train.merge(store, on="Store", how="left", validate="many_to_one")

# Actual trading days only (open & non-zero sales) — consistent with project 08.
actual = master[(master["Open"] == 1) & (master["Sales"] > 0)].copy()

print(f"Trading-day rows: {len(actual):,} of {len(master):,} "
      f"({len(actual)/len(master):.0%})")
print("Actual net sales by year (EUR million):")
print((actual.groupby("Year")["Sales"].sum() / 1e6).round(1).to_string())

## 3 · Building the Plan (Baseline + Growth Target)

**Planning rule (documented, reproducible):**

> Plan for store *s* in month *m* of year *Y* = *s*'s **average daily sales in
> the same month of year Y-1**, uplifted by the assumed **growth target** (+3 %).
> `PlanDaily(s, Y, m) = mean(open-day sales of s in month m of Y-1) × (1 + GROWTH_TARGET)`

This mirrors how a real *Bereichscontrolling* sets a first budget pass: take last
year's run-rate per unit and apply a corporate growth target. Using the **same
month** of the prior year makes the plan **seasonal** — so the monthly variance
reflects genuine performance deviations rather than the (large) seasonal swing
between January and December. The daily plan is then attached to **each actual
trading day** of the plan year, so the plan sum scales with the store's realised
trading days: the variance isolates *daily sales performance vs. target*, not
differences in how many days a store happened to be open.

Baselines: **2013 → plans 2014**, **2014 → plans 2015**. All 1,115 stores have
prior-year history in every month, so every store/month gets a plan.

In [ ]:
# --- Per-store, per-year, per-month baseline (seasonal avg. daily sales) -----
baseline = (actual.groupby(["Store", "Year", "Month"])["Sales"].mean()
            .rename("BaselineDaily").reset_index())
baseline["Year"] = baseline["Year"] + 1                 # this baseline plans the NEXT year
baseline["PlanDaily"] = baseline["BaselineDaily"] * (1 + GROWTH_TARGET)
plan_lookup = baseline[["Store", "Year", "Month", "PlanDaily"]]

# --- Attach the daily plan to every actual trading day in the plan years -----
pa = (actual[actual["Year"].isin(PLAN_YEARS)]
      .merge(plan_lookup, on=["Store", "Year", "Month"], how="left"))
missing = pa["PlanDaily"].isna().sum()
print(f"Plan years {PLAN_YEARS}: {len(pa):,} trading days, "
      f"{missing} without a prior-year baseline")
pa = pa.dropna(subset=["PlanDaily"]).copy()

pa["PlanSales"]   = pa["PlanDaily"]                      # one planned trading day
pa["ActualSales"] = pa["Sales"]
pa["VarianceAbs"] = pa["ActualSales"] - pa["PlanSales"]

# --- Reality check: realised YoY growth vs. the +3 % target ------------------
avg_daily = actual.groupby("Year")["Sales"].mean()
realised_2014 = (avg_daily[2014] / avg_daily[2013] - 1) * 100
print(f"\nRealised avg-daily-sales growth 2014 vs 2013: {realised_2014:+.1f}% "
      f"(planning target was +{GROWTH_TARGET:.0%})")
print("(2015 is year-to-date Jan-Jul, so a full-year YoY figure would be "
      "seasonally biased and is not shown here.)")

**Finding — plan realism:** the assumed **+3 %** target is deliberately close
to the store base's *realised* 2014-vs-2013 run-rate growth, so the plan is
ambitious-but-plausible rather than a straw man. Stores then split into over- and
under-performers against it (§4).

## 4 · Variance Analysis (absolute & %)

`Variance = Actual − Plan`, reported in euros and as a percentage of plan, at
three levels: **month**, **store type**, and **individual store**.

In [ ]:
def variance_block(df, keys):
    g = (df.groupby(keys)
           .agg(PlanSales=("PlanSales", "sum"),
                ActualSales=("ActualSales", "sum"),
                TradingDays=("ActualSales", "size"))
           .reset_index())
    g["VarianceAbs"] = g["ActualSales"] - g["PlanSales"]
    g["VariancePct"] = np.where(g["PlanSales"] > 0,
                                g["VarianceAbs"] / g["PlanSales"] * 100, np.nan)
    return g

var_month = variance_block(pa, ["YearMonth", "Year"]).sort_values("YearMonth")
var_type  = variance_block(pa, ["StoreType", "Year"]).sort_values(["Year", "StoreType"])
var_store = variance_block(pa, ["Store", "StoreType", "Year"])

var_month.round(2).to_csv(f"{OUTPUT_DIR}/plan_actual_monthly.csv", index=False, encoding="utf-8-sig")
var_type.round(2).to_csv(f"{OUTPUT_DIR}/variance_by_storetype.csv", index=False, encoding="utf-8-sig")
var_store.round(2).to_csv(f"{OUTPUT_DIR}/plan_actual_by_store.csv", index=False, encoding="utf-8-sig")

tot_plan = pa["PlanSales"].sum()
tot_act  = pa["ActualSales"].sum()
print(f"Total plan   : EUR {tot_plan/1e6:,.1f} m")
print(f"Total actual : EUR {tot_act/1e6:,.1f} m")
print(f"Overall variance: {(tot_act-tot_plan):,.0f} EUR "
      f"({(tot_act-tot_plan)/tot_plan*100:+.1f} %)\n")
print("Variance by store type (EUR m plan / actual / var %):")
print(var_type.assign(PlanM=lambda d: (d.PlanSales/1e6).round(1),
                      ActM=lambda d: (d.ActualSales/1e6).round(1),
                      VarPct=lambda d: d.VariancePct.round(1))
      [["Year", "StoreType", "PlanM", "ActM", "VarPct"]].to_string(index=False))

**Finding — variance:** aggregated over the plan window the store base lands
close to plan, but the **spread between stores is the controlling story** — the
per-store file (`plan_actual_by_store.csv`) and the dashboard surface the top
over- and under-performers a *Bereichscontroller* would follow up on.

## 5 · Cost & Profit Model (Contribution-Margin Accounting)

All euro figures below are **model assumptions** (§0). We use a two-tier
*Deckungsbeitragsrechnung*:

| Step | Definition |
|------|------------|
| Variable cost | `Sales × variable-cost-ratio[StoreType]` |
| **Contribution margin (DB I)** | `Sales − variable cost` |
| Fixed cost | `€900 × open trading days` |
| **Operating profit** | `Contribution margin − fixed cost` |

Because fixed cost is flat per store, low-revenue stores can fall **below
break-even** — surfacing them is exactly the kind of profitability screening
controlling is asked for.

In [ ]:
pnl = pa.copy()
pnl["VarCostRatio"]       = pnl["StoreType"].map(VAR_COST_RATIO)
pnl["VariableCost"]       = pnl["ActualSales"] * pnl["VarCostRatio"]
pnl["FixedCost"]          = float(FIXED_COST_PER_OPEN_DAY)      # one open trading day
pnl["ContributionMargin"] = pnl["ActualSales"] - pnl["VariableCost"]
pnl["OperatingProfit"]    = pnl["ContributionMargin"] - pnl["FixedCost"]

def pnl_block(df, keys):
    g = (df.groupby(keys)
           .agg(Revenue=("ActualSales", "sum"),
                VariableCost=("VariableCost", "sum"),
                FixedCost=("FixedCost", "sum"),
                ContributionMargin=("ContributionMargin", "sum"),
                OperatingProfit=("OperatingProfit", "sum"),
                TradingDays=("ActualSales", "size"))
           .reset_index())
    g["TotalCost"]           = g["VariableCost"] + g["FixedCost"]
    g["CMRatioPct"]          = g["ContributionMargin"] / g["Revenue"] * 100
    g["OperatingMarginPct"]  = g["OperatingProfit"] / g["Revenue"] * 100
    return g

pnl_month = pnl_block(pnl, ["YearMonth"]).sort_values("YearMonth")
pnl_type  = pnl_block(pnl, ["StoreType"]).sort_values("StoreType")
pnl_store = pnl_block(pnl, ["Store", "StoreType"])

nstores = pnl.groupby("StoreType")["Store"].nunique().rename("Stores").reset_index()
pnl_type = pnl_type.merge(nstores, on="StoreType")

pnl_month.round(2).to_csv(f"{OUTPUT_DIR}/pnl_monthly.csv", index=False, encoding="utf-8-sig")
pnl_type.round(2).to_csv(f"{OUTPUT_DIR}/pnl_by_storetype.csv", index=False, encoding="utf-8-sig")
pnl_store.round(2).to_csv(f"{OUTPUT_DIR}/profit_by_store.csv", index=False, encoding="utf-8-sig")

loss_stores = int((pnl_store["OperatingProfit"] < 0).sum())
print(f"Revenue            : EUR {pnl['ActualSales'].sum()/1e6:,.1f} m")
print(f"Variable cost      : EUR {pnl['VariableCost'].sum()/1e6:,.1f} m")
print(f"Fixed cost         : EUR {pnl['FixedCost'].sum()/1e6:,.1f} m")
print(f"Contribution margin: EUR {pnl['ContributionMargin'].sum()/1e6:,.1f} m "
      f"({pnl['ContributionMargin'].sum()/pnl['ActualSales'].sum()*100:.1f} % of sales)")
print(f"Operating profit   : EUR {pnl['OperatingProfit'].sum()/1e6:,.1f} m "
      f"({pnl['OperatingProfit'].sum()/pnl['ActualSales'].sum()*100:.1f} % margin)")
print(f"\nLoss-making stores under the assumed model: "
      f"{loss_stores} of {pnl_store.shape[0]}")
print("\nP&L by store type:")
print(pnl_type[["StoreType", "Stores", "Revenue", "ContributionMargin",
                "OperatingProfit", "OperatingMarginPct"]]
      .assign(Revenue=lambda d: (d.Revenue/1e6).round(1),
              ContributionMargin=lambda d: (d.ContributionMargin/1e6).round(1),
              OperatingProfit=lambda d: (d.OperatingProfit/1e6).round(1),
              OperatingMarginPct=lambda d: d.OperatingMarginPct.round(1))
      .to_string(index=False))

**Finding — profitability:** under the assumed cost model the operating
margin sits in a realistic single-digit retail range, and margin **differs by
store type** because the fixed-cost hurdle is easier to clear at higher daily
revenue (operating leverage). A handful of low-revenue stores fall below
break-even under these assumptions — the exact list is in `profit_by_store.csv`.

## 6 · KPI Cockpit & Assumptions Export

In [ ]:
rev  = pnl["ActualSales"].sum()
varc = pnl["VariableCost"].sum()
fixc = pnl["FixedCost"].sum()
cm   = pnl["ContributionMargin"].sum()
opr  = pnl["OperatingProfit"].sum()

kpis = {
    "Actual Revenue (EUR million)":        round(rev / 1e6, 1),
    "Plan Revenue (EUR million)":          round(tot_plan / 1e6, 1),
    "Revenue Variance (%)":                round((rev - tot_plan) / tot_plan * 100, 1),
    "Total Cost (EUR million)":            round((varc + fixc) / 1e6, 1),
    "Contribution Margin (EUR million)":   round(cm / 1e6, 1),
    "Operating Profit (EUR million)":      round(opr / 1e6, 1),
    "Operating Margin (%)":                round(opr / rev * 100, 1),
    "Loss-making Stores (Count)":          loss_stores,
}
kpi = pd.DataFrame({"KPI": list(kpis), "Value": list(kpis.values())})
kpi.to_csv(f"{OUTPUT_DIR}/kpi_controlling.csv", index=False, encoding="utf-8-sig")

assumptions = pd.DataFrame(ASSUMPTIONS, columns=["Parameter", "Value", "Note"])
assumptions.to_csv(f"{OUTPUT_DIR}/assumptions.csv", index=False, encoding="utf-8-sig")

print("Exported to ./output:")
for f in ["kpi_controlling.csv", "assumptions.csv", "plan_actual_monthly.csv",
          "variance_by_storetype.csv", "plan_actual_by_store.csv",
          "pnl_monthly.csv", "pnl_by_storetype.csv", "profit_by_store.csv"]:
    print("  -", f)
print()
display(kpi)
display(assumptions)

## 7 · Summary, Interpretation & Limitations

**What this shows (method, on real sales data):**
- A reproducible **plan/actual** framework (baseline + growth target) at day,
  month, store and store-type level.
- **Variance analysis** in € and %, with over-/under-performer ranking.
- A **contribution-margin** cost/profit model (DB I → operating profit) with a
  break-even screen across stores.
- A KPI cockpit feeding an interactive dashboard (`dashboard/`).

**Limitations (read alongside every figure):**
1. **No real financials.** The dataset has no budget/cost/margin data. The plan
   is a derived rule; all cost/profit euros are **assumptions**, not Rossmann
   actuals.
2. **Plan design choice.** A prior-year-run-rate + flat growth target is one of
   several valid budgeting methods (others: moving average, seasonal/top-down,
   bottom-up). Results depend on this choice.
3. **Flat growth target** ignores store-specific potential, cannibalisation and
   competition dynamics.
4. **Cost model is simplified**: one variable ratio per store type + one flat
   daily fixed cost. Real cost structures vary by location, wage level, lease
   and promotion intensity.
5. **2015 is year-to-date** (Jan-Jul); annual figures for 2015 are partial.
6. Scope excludes closed days and zero-sales days, consistent with project 08.

**How to reproduce / stress-test:** every assumption lives in the `ASSUMPTIONS`
block in §0. Change a number, re-run, and the CSVs + dashboard update — which is
the honest way to present modelled figures: transparent inputs, reproducible
outputs.